<a href="https://colab.research.google.com/github/catarina1532/pml/blob/main/petrignano.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Comparing two approaches for cross-validation: the standard KFold and TimeSeriesSplit using the Petrignano dataset

##0. Data preprocessing
**Task:** Organize data; deal with missing values; determine X,y sorted chronologically; (optional) add seasonality variable

*prompt:* Load a time series dataset, parse dates, sort chronologically, handle missing values using time-based interpolation, create lagged features (previous 2 time steps) for all variables including the target, and add a seasonal month feature.

*prompt:* The date format in 'Date' column is DD/MM/YYYY

df = df.fillna(method='bfill')

FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use bfill() instead.

In [28]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("Petrignano.csv")

# Display the first 5 rows of the DataFrame
display(df.head())

,Date,Rainfall_Bastia_Umbra,Depth_to_Groundwater_P25,Temperature_Bastia_Umbra,Temperature_Petrignano,Volume_C10_Petrignano,Hydrometry_Fiume_Chiascio_Petrignano
0,1/1/2009,0.0,-31.14,5.2,4.9,-24530.688,2.4
1,2/1/2009,0.0,-31.11,2.3,2.5,-28785.888,2.5
2,3/1/2009,0.0,-31.07,4.4,3.9,-25766.208,2.4
3,4/1/2009,0.0,-31.05,0.8,0.8,-27919.296,2.4
4,5/1/2009,0.0,-31.01,-1.9,-2.1,-29854.656,2.3


In [29]:
# Convert Date column
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values('Date')

# Set Date as index
df = df.set_index('Date')

# Handle missing values
df = df.interpolate(method='time')  # time-based interpolation
df = df.bfill() #fallback if needed (instead of fillna)

# Seasonality feature
df['month'] = df.index.month # time-related feature engineering

# Create lagged features (t-1 and t-2 days) ---
lags = [1, 2]
df_lagged = df.copy()

for lag in lags:
    for col in df.columns:
        df_lagged[f"{col}_lag{lag}"] = df[col].shift(lag)

# Drop rows with NaN caused by lagging
df_lagged = df_lagged.dropna()

# Define the target variable (y) and features (X)
y = df_lagged['Depth_to_Groundwater_P25']
X = df_lagged.drop(columns=['Depth_to_Groundwater_P25'])

# Ensure X and y are pre-sorted chronologically
X = X.sort_index()
y = y.sort_index()

# Display the first few rows and information of the processed DataFrame
display(df.head())
df.info()

,Rainfall_Bastia_Umbra,Depth_to_Groundwater_P25,Temperature_Bastia_Umbra,Temperature_Petrignano,Volume_C10_Petrignano,Hydrometry_Fiume_Chiascio_Petrignano,month
Date,,,,,,,
2009-01-01,0.0,-31.14,5.2,4.9,-24530.688,2.4,1
2009-01-02,0.0,-31.11,2.3,2.5,-28785.888,2.5,1
2009-01-03,0.0,-31.07,4.4,3.9,-25766.208,2.4,1
2009-01-04,0.0,-31.05,0.8,0.8,-27919.296,2.4,1
2009-01-05,0.0,-31.01,-1.9,-2.1,-29854.656,2.3,1


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 4199 entries, 2009-01-01 to 2020-06-30
Data columns (total 7 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Rainfall_Bastia_Umbra                 4199 non-null   float64
 1   Depth_to_Groundwater_P25              4199 non-null   float64
 2   Temperature_Bastia_Umbra              4199 non-null   float64
 3   Temperature_Petrignano                4199 non-null   float64
 4   Volume_C10_Petrignano                 4199 non-null   float64
 5   Hydrometry_Fiume_Chiascio_Petrignano  4199 non-null   float64
 6   month                                 4199 non-null   int32  
dtypes: float64(6), int32(1)
memory usage: 246.0 KB


This step prepares the dataset for a time series regression problem.

The response variable is Depth_to_Groundwater_P25.
The predictors include all original variables and their values at the previous two time steps (lags 1 and 2).

Missing values were handled using time-based interpolation, which is appropriate because groundwater and environmental variables tend to evolve smoothly over time.

Lagged variables were created to convert the time series into a supervised learning problem, where the value at time t is predicted using values at t−1 and t−2.

A seasonality variable (month) was added to capture recurring yearly patterns.

Including lagged versions of the target variable introduces strong temporal autocorrelation, which can make short-term predictions much easier and may lead to very high predictive performance.

##1. The hard split

*prompt:* Split time series data into train and test sets without shuffling, keeping chronological order. I want to reserve 20% of the data for testing

In [22]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets (80% train, 20% test)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
) # without shuffling

print(f"Shape of X_train_full: {X_train_full.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train_full: {y_train_full.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train_full: (3357, 20)
Shape of X_test: (840, 20)
Shape of y_train_full: (3357,)
Shape of y_test: (840,)


This step creates an independent test data set representing future observations.

The split is performed without shuffling, ensuring that training data comes strictly from the past and test data represents unseen future values.

This is essential in time series because:
- shuffling would mix past and future
- leading to data leakage and unrealistic performance estimates

The test set simulates a real-world scenario where we predict future groundwater levels using only past information.

##2. Define the two CV strategies

*prompt:* Define two cross-validation strategies for time series comparison: a shuffled KFold and a TimeSeriesSplit

In [23]:
from sklearn.model_selection import KFold, TimeSeriesSplit

cv_naive = KFold(n_splits=5, shuffle=True, random_state=42)
cv_temporal = TimeSeriesSplit(n_splits=5)

Two cross-validation strategies are defined:

- Naive KFold (shuffled) randomly splits the data into folds, ignores temporal order, and leads to data leakage since future observations can be used to predict the past.

- TimeSeriesSplit (temporal)
preserves chronological order, trains on past data and tests on future data, and provides a more realistic evaluation of model performance.

The purpose of this comparison is to demonstrate how improper validation (KFold) can produce overly optimistic results in time series problems.

##3. Experiment with a fixed model

*prompt:* Create a pipeline with StandardScaler and DecisionTreeRegressor and evaluate model performance using cross_val_score with R2 under both KFold and TimeSeriesSplit strategies

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', DecisionTreeRegressor(max_depth=10, random_state=42))
])

# Strategy 1: Naive (Shuffled)
# This will likely report a very high R2 because of leakage.
scores_naive = cross_val_score(pipe, X_train_full, y_train_full, cv=cv_naive, scoring='r2')

# Strategy 2: Temporal (Sequential)
# This will report a lower, more honest R2.
scores_temporal = cross_val_score(pipe, X_train_full, y_train_full, cv=cv_temporal, scoring='r2')

print(f"Naive CV R2:    {scores_naive.mean():.4f} (+/- {scores_naive.std():.4f})")
print(f"Temporal CV R2: {scores_temporal.mean():.4f} (+/- {scores_temporal.std():.4f})")

# Train on all training data and test on the unseen "future"
pipe.fit(X_train_full, y_train_full)
final_test_r2 = r2_score(y_test, pipe.predict(X_test))

print(f"\nActual Test R2: {final_test_r2:.4f}")

Naive CV R2:    0.9994 (+/- 0.0000)
Temporal CV R2: 0.6080 (+/- 0.5121)

Actual Test R2: 0.9821


A pipeline was used to standardize features and train a Decision Tree regressor.

- Naive CV (KFold): 0.9994 -> extremely high
- TimeSeriesSplit: 0.6080 (± 0.5121) -> much lower and highly variable
- Test R²: 0.9821 -> very high

The Naive KFold score is misleadingly high due to data leakage, as it mixes past and future observations.

The TimeSeriesSplit score is much lower and unstable, indicating that model performance varies significantly across different time periods.

The test score is unexpectedly very high, which is explained by the inclusion of lagged values of the target variable.

The model is effectively learning a persistence pattern, where groundwater levels at time t are strongly predicted by values at t−1 and t−2.
This strong autocorrelation makes short-term predictions easier and explains the high test performance.

##4. Model selection and evaluation

*prompt:* Implement a function using GridSearchCV with a pipeline including StandardScaler and DecisionTreeRegressor, tuning max_depth and evaluating performance using a specified cross-validation strategy

In [25]:
from sklearn.model_selection import GridSearchCV

def evaluate_model_selection(X_train, y_train, X_test, y_test, cv_strategy, name):

    # STEP A: Pipeline
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', DecisionTreeRegressor(random_state=42))
    ])

    # STEP B: Hyperparameter grid
    param_grid = {
        'regressor__max_depth': [3, 5, 10, 15, 20]
    }

    # STEP C: GridSearch
    grid = GridSearchCV(
        pipe,
        param_grid,
        cv=cv_strategy,
        scoring='r2',
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    # STEP D: Final evaluation
    y_pred = grid.predict(X_test)
    test_r2 = r2_score(y_test, y_pred)

    print(f"\n===== Results for: {name} =====")
    print(f"Best Parameters: {grid.best_params_}")
    print(f"Internal CV R2: {grid.best_score_:.4f}")
    print(f"Test R2: {test_r2:.4f}")

    return grid

This step performs model selection using GridSearchCV.

- Hyperparameter tuned: max_depth of the Decision Tree
- Evaluation metric: R²
- Validation strategies: KFold and TimeSeriesSplit

Both methods selected the same optimal parameter (max_depth = 10). However, their internal CV scores differ significantly.

Naive KFold CV score (0.9994) is artificially inflated due to data leakage.

TimeSeriesSplit CV score (0.6080) reflects the true variability and difficulty of the prediction task.

Even though both methods select the same model, the reliability of the evaluation differs, with TimeSeriesSplit providing a more trustworthy estimate.

##5. Running the comparison

*prompt:* Run model selection using both KFold and TimeSeriesSplit and compare results

In [26]:
# Call the function twice and compare the gaps between CV and Test scores.
result_naive = evaluate_model_selection(
    X_train_full, y_train_full, X_test, y_test,
    cv_naive, "Naive K-Fold"
)

result_temporal = evaluate_model_selection(
    X_train_full, y_train_full, X_test, y_test,
    cv_temporal, "TimeSeriesSplit"
)


===== Results for: Naive K-Fold =====
Best Parameters: {'regressor__max_depth': 10}
Internal CV R2: 0.9994
Test R2: 0.9821

===== Results for: TimeSeriesSplit =====
Best Parameters: {'regressor__max_depth': 10}
Internal CV R2: 0.6080
Test R2: 0.9821


**Conclusion:**

Naive KFold:
- Very high CV score (0.9994)
- Slightly lower test score (0.9821)
- Overestimates performance due to data leakage

TimeSeriesSplit:
- Lower and highly variable CV score (0.6080 ± 0.5121)
- Test score much higher than CV
- Reveals true variability across time, but underestimates performance in this case

KFold produces unrealistically high performance due to data leakage, while TimeSeriesSplit provides a more reliable estimate by preserving temporal order. The high test performance is explained by strong autocorrelation in the target variable, making short-term predictions easier. Therefore, TimeSeriesSplit is the most appropriate evaluation method.